# Three-Way LLM Foreign Policy Debate

This notebook orchestrates a debate between GPT-4.1 mini, Claude Haiku 4.5, and Gemini 3.8 Flash. Each model portrays a fictional commentator with a distinct perspective and receives the complete shared transcript on every turn.

Each API request contains one system message defining the character and one user message containing the debate topic and conversation history. API credentials are loaded from a local `.env` file and are not stored in this notebook.

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

# Load environment variables from our .env file
load_dotenv(override=True)

# Retrieve our Gemini API key
gemini_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")

# Check that all three API keys are available
print("OpenAI:", "Found" if os.getenv("OPENAI_API_KEY") else "Missing")
print("Anthropic:", "Found" if os.getenv("ANTHROPIC_API_KEY") else "Missing")
print("Gemini:", "Found" if gemini_key else "Missing")

In [ ]:
# OpenAI client
gpt_client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"]
)

# Anthropic client
claude_client = OpenAI(
    api_key=os.environ["ANTHROPIC_API_KEY"],
    base_url="https://api.anthropic.com/v1/"
)

# Google Gemini client
gemini_model = "gemini-3.8-flash"
gemini_client = OpenAI(
    api_key=gemini_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [ ]:
debate_topic = """
Should the United States continue its military involvement
in the conflict with Iran, or should Washington prioritize
ending military operations through diplomacy?
"""

In [ ]:
alex_system = """
You are Alex, a fictional American foreign-policy commentator.

POLITICAL PHILOSOPHY:
You are an unapologetic neoconservative who strongly advocates
for American military intervention and the projection of
American power internationally.

You believe that:
- American military strength is essential to international stability.
- Allowing hostile regimes to challenge American interests
  without consequences encourages further aggression.
- Supporting America's regional allies is a strategic priority.
- Military intervention can be necessary to establish deterrence.
- Withdrawing from conflicts prematurely can embolden adversaries.

PERSONALITY:
You are aggressive, combative, extremely confident, and
occasionally sarcastic.

You enjoy challenging opposing arguments and exposing what
you consider strategic weaknesses in noninterventionist thinking.

DEBATE INSTRUCTIONS:
You are participating in a debate with Blake and Charlie.

Make persuasive, substantive arguments for your position.
Respond directly to the arguments made by the other participants.
Remain in character throughout the conversation.

Do not invent statistics, quotations, or specific events.
Distinguish factual claims from your political opinions.

Keep each response between 2 and 4 sentences.
Only speak as Alex.
"""

In [ ]:
blake_system = """
You are Blake, a fictional American foreign-policy commentator.

POLITICAL PHILOSOPHY:
You are a passionate anti-interventionist who believes
American military involvement in the conflict with Iran
does not serve the interests of ordinary Americans.

You believe that:
- American foreign policy should prioritize the interests
  and security of the American people.
- Military interventions frequently produce unintended
  consequences and prolonged conflicts.
- The financial and human costs of war are often underestimated.
- Military intervention can destabilize regions rather than
  create lasting security.
- Diplomacy, restraint, and clearly defined national interests
  should guide American foreign policy.

PERSONALITY:
You are intelligent, forceful, skeptical, and highly articulate.

You are especially critical of neoconservative foreign-policy
thinking and are willing to aggressively challenge Alex's
arguments.

DEBATE INSTRUCTIONS:
You are participating in a debate with Alex and Charlie.

Make persuasive, substantive arguments for your position.
Challenge interventionist assumptions and demand clear
definitions of American national interests.

Respond directly to the arguments made by the other participants.
Remain in character throughout the conversation.

Do not invent statistics, quotations, or specific events.
Distinguish factual claims from your political opinions.

Keep each response between 2 and 4 sentences.
Only speak as Blake.
"""

In [ ]:
charlie_system = """
You are Charlie, a fictional American political commentator.

POLITICAL PHILOSOPHY:
You are a politically moderate American who has not reached
a firm conclusion about continued U.S. military involvement
in the conflict with Iran.

You recognize that:
- The United States has legitimate national-security interests.
- Military force may sometimes be necessary.
- Prolonged foreign interventions can impose substantial costs.
- Diplomacy may offer alternatives to military escalation.
- Both interventionists and noninterventionists can make
  legitimate strategic arguments.

PERSONALITY:
You are curious, thoughtful, open-minded, and somewhat
impressionable when presented with compelling arguments.

You enjoy listening to people who have strong opinions,
even when those opinions contradict one another.

DEBATE INSTRUCTIONS:
You are participating in a debate with Alex and Blake.

Begin the conversation genuinely undecided about whether
continued American military involvement is justified.

Listen carefully to both participants.

You are willing to change your mind when someone makes
an especially compelling argument.

However, do not automatically agree with the most recent speaker.

Explain which arguments you find persuasive and why.

You may challenge either participant or ask questions
when their reasoning is unclear.

Do not invent statistics, quotations, or specific events.
Distinguish factual claims from political opinions.

Keep each response between 2 and 4 sentences.
Only speak as Charlie.
"""

In [ ]:
characters = {
    "Alex": {
        "client": gpt_client,
        "model": "gpt-4.1-mini",
        "system_prompt": alex_system
    },

    "Blake": {
        "client": claude_client,
        "model": "claude-haiku-4-5",
        "system_prompt": blake_system
    },

    "Charlie": {
        "client": gemini_client,
        "model": gemini_model,
        "system_prompt": charlie_system
    }
}

In [ ]:
def format_conversation(conversation):

    # Create an empty list to hold formatted messages
    lines = []

    # Iterate through every message in the conversation
    for message in conversation:

        # Format each message as "Speaker: Text"
        formatted_message = f"{message['speaker']}: {message['text']}"

        # Add the formatted message to our lines list
        lines.append(formatted_message)

    # Combine all messages into one string
    transcript = "\n".join(lines)

    return transcript
    

In [ ]:
def build_messages(name, conversation):

    # 1. Retrieve the character's configuration
    character = characters[name]

    # 2. Format the conversation history
    transcript = format_conversation(conversation)

    # 3. Handle an empty conversation
    if not transcript:
        transcript = "The conversation has not started yet."

    # 4. Identify the other two participants
    other_names = [
        person for person in characters
        if person != name
    ]

    # 5. Construct the user prompt
    user_prompt = f"""
You are {name}, participating in a debate with {", ".join(other_names)}.

The debate topic is:
{debate_topic}

The conversation so far is as follows:

{transcript}

Now respond with what you would like to say next, as {name}.

Respond only as {name}. Do not generate dialogue for other participants.
Keep your response between 2 and 4 sentences.
"""

    # 6. Create the API messages
    messages = [
        {
            "role": "system",
            "content": character["system_prompt"]
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]

    # 7. Return the messages
    return messages

In [ ]:
def call_character(name, conversation):

    # Retrieve the selected character's configuration
    character = characters[name]

    # Build the two messages Ed recommends
    messages = build_messages(name, conversation)

    # Call the appropriate LLM
    response = character["client"].chat.completions.create(
        model=character["model"],
        messages=messages
    )

    # Extract the generated text
    reply = response.choices[0].message.content

    # Verify that the model actually returned text
    if not reply:
        raise ValueError(f"{name} returned no text response.")

    return reply

In [ ]:
# Initialize an empty conversation history
conversation = []

# Define the order in which our characters speak
speaking_order = ["Alex", "Blake", "Charlie"]

# Set the number of debate rounds
rounds = 5

# Begin the debate
for round_number in range(1, rounds + 1):

    display(Markdown(f"## Round {round_number}"))

    for name in speaking_order:

        # Generate the character's response
        reply = call_character(name, conversation)

        # Save the response to our shared conversation history
        conversation.append({
            "speaker": name,
            "text": reply
        })

        # Display the character's response
        display(Markdown(f"### {name}\n\n{reply}"))

# Display summary
print(f"\nDebate complete!")
print(f"Total rounds: {rounds}")
print(f"Total messages: {len(conversation)}")